# Exploratory Data Analysis (EDA) - Autoscaling Analysis

Phân tích chi tiết dữ liệu HTTP logs:
- Phân phối requests & error rates
- Time series patterns (hourly, daily, weekly)
- Outliers & gaps detection
- Feature correlations
- Spike vs DDoS comparison
- Data quality validation

## 1. Import Libraries & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Libraries loaded successfully")

## 2. Load Cleaned Data

In [ ]:
# Load cleaned training data
try:
    df = pd.read_csv('DATA/clean_data_train.csv')
    print(f"✓ Loaded {len(df):,} records from clean_data_train.csv")
except:
    print("⚠ clean_data_train.csv not found. Running data loader...")
    from src.data_loader import load_and_prepare_data
    train_data, _, _ = load_and_prepare_data('DATA/train.txt')
    df = train_data['5min'].reset_index()
    print(f"✓ Loaded {len(df):,} aggregated records (5min window)")

# Display basic info
print(f"\nDataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")

## 3. Data Quality Overview

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

print("Missing Values:")
print(missing[missing > 0] if missing.sum() > 0 else "✓ No missing values")

# Basic statistics
print("\n" + "="*60)
print("Basic Statistics:")
print("="*60)
print(df.describe())

# Data quality checks
print("\n" + "="*60)
print("Data Quality Metrics:")
print("="*60)
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    print(f"Date Range: {df['timestamp'].min()} → {df['timestamp'].max()}")
    print(f"Duration: {(df['timestamp'].max() - df['timestamp'].min()).days} days")

if 'requests' in df.columns:
    print(f"\nRequests Statistics:")
    print(f"  Mean: {df['requests'].mean():.0f}")
    print(f"  Median: {df['requests'].median():.0f}")
    print(f"  Min: {df['requests'].min():.0f}")
    print(f"  Max: {df['requests'].max():.0f}")
    print(f"  Std: {df['requests'].std():.0f}")

if 'error_rate' in df.columns:
    print(f"\nError Rate Statistics:")
    print(f"  Mean: {df['error_rate'].mean():.4f} ({df['error_rate'].mean()*100:.2f}%)")
    print(f"  Median: {df['error_rate'].median():.4f}")
    print(f"  Min: {df['error_rate'].min():.4f}")
    print(f"  Max: {df['error_rate'].max():.4f}")

## 4. Distribution Analysis

In [ ]:
# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Requests Distribution
axes[0, 0].hist(df['requests'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df['requests'].mean(), color='red', linestyle='--', label=f'Mean: {df["requests"].mean():.0f}')
axes[0, 0].axvline(df['requests'].median(), color='green', linestyle='--', label=f'Median: {df["requests"].median():.0f}')
axes[0, 0].set_xlabel('Requests')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Request Distribution')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Error Rate Distribution
axes[0, 1].hist(df['error_rate'], bins=50, color='salmon', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(df['error_rate'].mean(), color='red', linestyle='--', label=f'Mean: {df["error_rate"].mean():.4f}')
axes[0, 1].set_xlabel('Error Rate')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Error Rate Distribution')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Requests Boxplot
axes[1, 0].boxplot(df['requests'], vert=True)
axes[1, 0].set_ylabel('Requests')
axes[1, 0].set_title('Request Outliers (Boxplot)')
axes[1, 0].grid(True, alpha=0.3)

# 4. Error Rate Boxplot
axes[1, 1].boxplot(df['error_rate'], vert=True)
axes[1, 1].set_ylabel('Error Rate')
axes[1, 1].set_title('Error Rate Outliers (Boxplot)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Distribution analysis complete")

## 5. Outlier Detection

In [ ]:
# Outlier detection using IQR method
def detect_outliers_iqr(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return (data < lower_bound) | (data > upper_bound)

# Detect outliers
outliers_requests = detect_outliers_iqr(df['requests'])
outliers_error = detect_outliers_iqr(df['error_rate'])

print("Outlier Detection Results:")
print(f"Requests Outliers: {outliers_requests.sum():,} ({outliers_requests.sum()/len(df)*100:.2f}%)")
print(f"Error Rate Outliers: {outliers_error.sum():,} ({outliers_error.sum()/len(df)*100:.2f}%)")

# Analyze outlier characteristics
if outliers_requests.sum() > 0:
    print(f"\nRequest Outlier Stats:")
    print(f"  Min: {df[outliers_requests]['requests'].min():.0f}")
    print(f"  Max: {df[outliers_requests]['requests'].max():.0f}")
    print(f"  Mean: {df[outliers_requests]['requests'].mean():.0f}")

if outliers_error.sum() > 0:
    print(f"\nError Rate Outlier Stats:")
    print(f"  Min: {df[outliers_error]['error_rate'].min():.4f}")
    print(f"  Max: {df[outliers_error]['error_rate'].max():.4f}")
    print(f"  Mean: {df[outliers_error]['error_rate'].mean():.4f}")

## 6. Time Series Patterns

In [ ]:
# Ensure timestamp is datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Extract time features
df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.day
df['day_of_week'] = df['timestamp'].dt.dayofweek  # 0=Mon, 6=Sun
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# Hourly pattern
hourly_pattern = df.groupby('hour')['requests'].agg(['mean', 'std', 'count'])

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Hourly Pattern
axes[0, 0].plot(hourly_pattern.index, hourly_pattern['mean'], marker='o', color='blue', linewidth=2)
axes[0, 0].fill_between(hourly_pattern.index, 
                         hourly_pattern['mean'] - hourly_pattern['std'],
                         hourly_pattern['mean'] + hourly_pattern['std'],
                         alpha=0.3)
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('Avg Requests')
axes[0, 0].set_title('Hourly Load Pattern')
axes[0, 0].grid(True, alpha=0.3)

# 2. Daily Pattern
daily_pattern = df.groupby('day_of_week')['requests'].agg(['mean', 'std'])
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[0, 1].bar(days, daily_pattern['mean'], color='green', alpha=0.7, edgecolor='black')
axes[0, 1].errorbar(range(7), daily_pattern['mean'], yerr=daily_pattern['std'], 
                     fmt='none', color='black', capsize=5)
axes[0, 1].set_xlabel('Day of Week')
axes[0, 1].set_ylabel('Avg Requests')
axes[0, 1].set_title('Daily Load Pattern')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Weekday vs Weekend
weekday_data = [df[df['is_weekend']==0]['requests'], df[df['is_weekend']==1]['requests']]
axes[1, 0].boxplot(weekday_data, labels=['Weekday', 'Weekend'])
axes[1, 0].set_ylabel('Requests')
axes[1, 0].set_title('Weekday vs Weekend Load')
axes[1, 0].grid(True, alpha=0.3)

# 4. Time Series
axes[1, 1].plot(df['timestamp'][:1000], df['requests'][:1000], color='purple', linewidth=1)
axes[1, 1].set_xlabel('Timestamp')
axes[1, 1].set_ylabel('Requests')
axes[1, 1].set_title('Load Time Series (First 1000 records)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Time series patterns analyzed")

## 7. Error Rate Analysis

In [ ]:
# Error rate by hour
hourly_error = df.groupby('hour')['error_rate'].agg(['mean', 'std', 'max'])

# Error rate by day of week
daily_error = df.groupby('day_of_week')['error_rate'].agg(['mean', 'std', 'max'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Hourly Error Pattern
axes[0].plot(hourly_error.index, hourly_error['mean'], marker='o', color='red', linewidth=2, label='Mean')
axes[0].plot(hourly_error.index, hourly_error['max'], marker='s', color='darkred', linewidth=1.5, label='Max', alpha=0.7)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Error Rate')
axes[0].set_title('Hourly Error Rate Pattern')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Daily Error Pattern
axes[1].bar(days, daily_error['mean'], color='orange', alpha=0.7, edgecolor='black', label='Mean')
axes[1].plot(range(7), daily_error['max'], marker='D', color='darkred', linewidth=2, markersize=8, label='Max')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Error Rate')
axes[1].set_title('Daily Error Rate Pattern')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Error Rate Summary:")
print(f"Overall Mean: {df['error_rate'].mean():.4f} ({df['error_rate'].mean()*100:.2f}%)")
print(f"Peak Hour: Hour {hourly_error['max'].idxmax()} with {hourly_error['max'].max():.4f}")
print(f"Peak Day: {days[daily_error['max'].idxmax()]} with {daily_error['max'].max():.4f}")

## 8. Spike Detection Analysis

In [ ]:
# Detect spikes (requests > 95th percentile)
spike_threshold = df['requests'].quantile(0.95)
df['is_spike'] = df['requests'] > spike_threshold

# Analyze spike characteristics
spikes = df[df['is_spike']]

print(f"Spike Detection Results:")
print(f"Threshold (95th percentile): {spike_threshold:.0f}")
print(f"Total Spikes: {spikes.shape[0]:,} ({spikes.shape[0]/len(df)*100:.2f}%)")
print(f"\nSpike Characteristics:")
print(f"  Avg Requests: {spikes['requests'].mean():.0f}")
print(f"  Max Requests: {spikes['requests'].max():.0f}")
print(f"  Avg Error Rate: {spikes['error_rate'].mean():.4f} ({spikes['error_rate'].mean()*100:.2f}%)")
print(f"  Max Error Rate: {spikes['error_rate'].max():.4f}")

# Spike timing
spike_by_hour = spikes['hour'].value_counts().sort_index()
spike_by_day = spikes['day_of_week'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Spikes by hour
axes[0].bar(spike_by_hour.index, spike_by_hour.values, color='red', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Number of Spikes')
axes[0].set_title('Spike Distribution by Hour')
axes[0].grid(True, alpha=0.3, axis='y')

# Spikes by day
spike_days = [spikes[spikes['day_of_week']==i].shape[0] for i in range(7)]
axes[1].bar(days, spike_days, color='darkred', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Number of Spikes')
axes[1].set_title('Spike Distribution by Day')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 9. Spike vs DDoS Pattern Comparison

In [ ]:
# Define DDoS pattern: high load + high error rate
load_threshold = df['requests'].quantile(0.90)
error_threshold = df['error_rate'].quantile(0.75)

df['potential_ddos'] = (df['requests'] > load_threshold) & (df['error_rate'] > error_threshold)

potential_ddos = df[df['potential_ddos']]

print("Pattern Comparison:")
print(f"\nSpikes (High Load Only):")
print(f"  Count: {df['is_spike'].sum()}")
print(f"  Avg Requests: {df[df['is_spike']]['requests'].mean():.0f}")
print(f"  Avg Error Rate: {df[df['is_spike']]['error_rate'].mean():.4f}")

print(f"\nPotential DDoS (High Load + High Error):")
print(f"  Count: {potential_ddos.shape[0]}")
print(f"  Avg Requests: {potential_ddos['requests'].mean():.0f}")
print(f"  Avg Error Rate: {potential_ddos['error_rate'].mean():.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scatter plot: Requests vs Error Rate
axes[0].scatter(df['requests'], df['error_rate'], alpha=0.3, s=20, label='Normal')
axes[0].scatter(df[df['is_spike']]['requests'], df[df['is_spike']]['error_rate'], 
                color='red', alpha=0.6, s=30, label='Spike')
axes[0].scatter(potential_ddos['requests'], potential_ddos['error_rate'], 
                color='darkred', alpha=0.8, s=50, marker='X', label='Potential DDoS')
axes[0].axvline(load_threshold, color='orange', linestyle='--', label='Load Threshold')
axes[0].axhline(error_threshold, color='orange', linestyle='--', label='Error Threshold')
axes[0].set_xlabel('Requests')
axes[0].set_ylabel('Error Rate')
axes[0].set_title('Spike vs DDoS Pattern (Requests vs Error Rate)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Time series with annotations
axes[1].plot(df['timestamp'][:2000], df['requests'][:2000], color='blue', linewidth=1, label='Requests')
spike_idx = df[df['is_spike']].index[:2000]
ddos_idx = potential_ddos[potential_ddos.index < 2000].index

if len(spike_idx) > 0:
    axes[1].scatter(df.loc[spike_idx, 'timestamp'], df.loc[spike_idx, 'requests'], 
                   color='red', s=50, alpha=0.7, label='Spike')
if len(ddos_idx) > 0:
    axes[1].scatter(df.loc[ddos_idx, 'timestamp'], df.loc[ddos_idx, 'requests'], 
                   color='darkred', s=100, marker='X', alpha=0.9, label='Potential DDoS')

axes[1].set_xlabel('Timestamp')
axes[1].set_ylabel('Requests')
axes[1].set_title('Time Series: Spike vs DDoS Detection (First 2000 records)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Feature Correlations

In [ ]:
# Select numeric columns for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove unnecessary columns
if 'is_spike' in numeric_cols:
    numeric_cols.remove('is_spike')
if 'potential_ddos' in numeric_cols:
    numeric_cols.remove('potential_ddos')
if 'is_weekend' in numeric_cols:
    numeric_cols.remove('is_weekend')
if 'day' in numeric_cols:
    numeric_cols.remove('day')

# Calculate correlation
if len(numeric_cols) > 1:
    corr_matrix = df[numeric_cols].corr()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
                square=True, linewidths=1, ax=ax, cbar_kws={"shrink": 0.8})
    plt.title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()
    
    print("Correlation Analysis:")
    print(corr_matrix)
else:
    print("Not enough numeric features for correlation analysis")

## 11. Gap Detection

In [ ]:
# Detect time gaps
df_sorted = df.sort_values('timestamp').reset_index(drop=True)
df_sorted['time_diff'] = df_sorted['timestamp'].diff()

# Define gap threshold (e.g., more than 10 minutes)
gap_threshold = pd.Timedelta(minutes=10)
gaps = df_sorted[df_sorted['time_diff'] > gap_threshold]

print(f"Time Gap Analysis:")
print(f"Gap Threshold: {gap_threshold}")
print(f"Number of Gaps: {len(gaps)}")
print(f"Total Missing Time: {gaps['time_diff'].sum()}")

if len(gaps) > 0:
    print(f"\nGap Statistics:")
    print(f"  Longest Gap: {gaps['time_diff'].max()}")
    print(f"  Shortest Gap: {gaps['time_diff'].min()}")
    print(f"  Mean Gap: {gaps['time_diff'].mean()}")
    
    # Show top gaps
    print(f"\nTop 5 Longest Gaps:")
    top_gaps = gaps.nlargest(5, 'time_diff')[['timestamp', 'time_diff']]
    for idx, (i, row) in enumerate(top_gaps.iterrows(), 1):
        print(f"  {idx}. {row['timestamp']}: {row['time_diff']}")
else:
    print("✓ No significant gaps detected")

## 12. Summary & Conclusions

In [ ]:
print("="*70)
print("EDA SUMMARY & KEY FINDINGS")
print("="*70)

print(f"\n1. DATA QUALITY:")
print(f"   ✓ Records: {len(df):,}")
print(f"   ✓ Missing values: {'None' if df.isnull().sum().sum() == 0 else 'Some'}")
print(f"   ✓ Duration: {(df['timestamp'].max() - df['timestamp'].min()).days} days")

print(f"\n2. LOAD CHARACTERISTICS:")
print(f"   ✓ Mean: {df['requests'].mean():.0f} requests")
print(f"   ✓ Peak: {df['requests'].max():.0f} requests")
print(f"   ✓ Std Dev: {df['requests'].std():.0f} (Coefficient of Variation: {df['requests'].std()/df['requests'].mean():.2f})")

print(f"\n3. ERROR PATTERNS:")
print(f"   ✓ Baseline Error Rate: {df['error_rate'].mean()*100:.2f}%")
print(f"   ✓ Peak Error Rate: {df['error_rate'].max()*100:.2f}%")
print(f"   ✓ High Error Periods: {(df['error_rate'] > df['error_rate'].quantile(0.90)).sum()} ({(df['error_rate'] > df['error_rate'].quantile(0.90)).sum()/len(df)*100:.2f}%)")

print(f"\n4. ANOMALIES DETECTED:")
print(f"   ✓ Spikes (High Load): {df['is_spike'].sum()} ({df['is_spike'].sum()/len(df)*100:.2f}%)")
print(f"   ✓ Potential DDoS (Load + Error): {potential_ddos.shape[0]} ({potential_ddos.shape[0]/len(df)*100:.2f}%)")
print(f"   ✓ Outlier Records: {outliers_requests.sum()} requests, {outliers_error.sum()} error rates")

print(f"\n5. TIME PATTERNS:")
print(f"   ✓ Peak Hour: {hourly_pattern['mean'].idxmax()}:00 ({hourly_pattern['mean'].max():.0f} requests)")
print(f"   ✓ Quietest Hour: {hourly_pattern['mean'].idxmin()}:00 ({hourly_pattern['mean'].min():.0f} requests)")
print(f"   ✓ Peak Day: {days[daily_pattern['mean'].idxmax()]} ({daily_pattern['mean'].max():.0f} requests)")
print(f"   ✓ Weekend Pattern: {'Lower than weekdays' if df[df['is_weekend']==1]['requests'].mean() < df[df['is_weekend']==0]['requests'].mean() else 'Higher than weekdays'}")

print(f"\n6. RECOMMENDATIONS FOR MODELING:")
print(f"   ✓ Feature Engineering: Use hour, day_of_week, is_weekend")
print(f"   ✓ Seasonality: Strong 24h pattern detected, consider SARIMA/Seasonal models")
print(f"   ✓ Ensemble: Recommend ensemble methods to capture non-linearities")
print(f"   ✓ Threshold Tuning: For scaling policy, use {df['requests'].quantile(0.75):.0f} (75th percentile) as baseline")
print(f"   ✓ DDoS Detection: Monitor correlation between load and error_rate")

print(f"\n" + "="*70)